# Implementa o pipeline declarativo
  #31

O objetivo aqui é comparar as duas abordagens lado a lado, não substituir a que já está em produção

## 1. Criar o arquivo de código em
`notebooks/notebooks_ldp_pipeline/orders_silver_declarative`

Versão anterior (antes das expectations), mantida aqui apenas como
referência histórica, não executada na segunda vez:
```
from pyspark import pipelines as dp
from pyspark.sql.functions import col


@dp.materialized_view()
def orders_silver_declarative():
    return (
        spark.read.table("olist_project.bronze.orders")  # noqa: F821
        .dropDuplicates(["order_id"])
        .filter(col("order_id").isNotNull())
        .filter(col("customer_id").isNotNull())
    )
```

# Adicionar quality expectations


In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col


@dp.materialized_view()
@dp.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dp.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL")
def orders_silver_declarative():
    return (
        spark.read.table("olist_project.bronze.orders")
        .dropDuplicates(["order_id"])
    )

**O que mudou em relação à versão anterior**

Antes, a checagem de nulo era feita com .filter() dentro da função, funcionava, mas era "silenciosa" (só removia, sem métrica). Agora, cada @dp.expect_or_drop("nome_da_regra", "condição SQL") declara a regra como metadado do pipeline: o framework aplica o filtro automaticamente e expõe, na tela do Pipeline, quantas linhas foram descartadas por qual regra especificamente.

Repare que tirei os .filter() manuais de dentro da função, as expectations substituem eles.

Depois de rodar

Na aba Expectations da tela do Pipeline (aquela coluna que apareceu como "-" no resultado anterior), você deve ver agora os nomes valid_order_id e valid_customer_id, com contagem de quantos registros passaram/foram descartados por cada uma.